# NLP Task 2 – Sentiment Analysis
**Author:** Neeraj  
**Dataset:** IMDb Movie Reviews  
**Pipeline:** Raw Data → Preprocessing → Feature Engineering → Model Training → Evaluation → Comparison

## Step 1 – Import Libraries

In [1]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

print('Libraries imported successfully!')

Libraries imported successfully!


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


## Step 2 – Load Dataset

In [2]:
import urllib.request

urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/Ankit152/IMDB-sentiment-analysis/master/IMDB-Dataset.csv",
    "IMDB_Dataset.csv"
)
print("Downloaded successfully!")

Downloaded successfully!


In [3]:
df = pd.read_csv("IMDB_Dataset.csv")
print("Shape:", df.shape)
df.head()

Shape: (50000, 2)


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


## Step 3 – Data Understanding

In [4]:
# Number of samples and class distribution
print("Total Samples:", len(df))
print("\nClass Distribution:")
print(df['sentiment'].value_counts())
print("\nMissing Values:")
print(df.isnull().sum())

Total Samples: 50000

Class Distribution:
sentiment
positive    25000
negative    25000
Name: count, dtype: int64

Missing Values:
review       0
sentiment    0
dtype: int64


In [5]:
# Sample texts
print("Sample Positive Review:")
print(df[df['sentiment']=='positive']['review'].iloc[0][:300])
print("\nSample Negative Review:")
print(df[df['sentiment']=='negative']['review'].iloc[0][:300])

Sample Positive Review:
One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Tru

Sample Negative Review:
Basically there's a family where a little boy (Jake) thinks there's a zombie in his closet & his parents are fighting all the time.<br /><br />This movie is slower than a soap opera... and suddenly, Jake decides to become Rambo and kill the zombie.<br /><br />OK, first of all when you're going to ma


## Step 4 – NLP Preprocessing

Steps applied:
1. Lowercase
2. Remove HTML tags
3. Remove special characters and punctuation
4. Tokenization
5. Remove stopwords
6. Lemmatization (alternative to Stemming)

In [6]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    # 1. Lowercase
    text = text.lower()
    # 2. Remove HTML tags
    text = re.sub(r'<.*?>', ' ', text)
    # 3. Remove special characters and punctuation
    text = re.sub(r'[^a-z\s]', ' ', text)
    # 4. Tokenization
    tokens = text.split()
    # 5. Remove stopwords + 6. Lemmatization
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words]
    return ' '.join(tokens)

# Test the function
sample = "This movie was ABSOLUTELY brilliant! <br /> Loved it!!!"
print("Original  :", sample)
print("Processed :", preprocess(sample))

Original  : This movie was ABSOLUTELY brilliant! <br /> Loved it!!!
Processed : movie absolutely brilliant loved


In [7]:
# Apply preprocessing to full dataset
df['clean'] = df['review'].apply(preprocess)
print("Preprocessing done!")
df[['review', 'clean', 'sentiment']].head(3)

Preprocessing done!


,review,clean,sentiment
0,One of the other reviewers has mentioned that ...,one reviewer mentioned watching oz episode hoo...,positive
1,A wonderful little production. <br /><br />The...,wonderful little production filming technique ...,positive
2,I thought this was a wonderful way to spend ti...,thought wonderful way spend time hot summer we...,positive


## Step 5 – Feature Engineering

In [8]:
# Encode labels: positive=1, negative=0
df['label'] = df['sentiment'].map({'positive': 1, 'negative': 0})

# Train-test split 80/20
X_train, X_test, y_train, y_test = train_test_split(
    df['clean'], df['label'], test_size=0.2, random_state=42
)

print("Train size:", len(X_train))
print("Test size :", len(X_test))

Train size: 40000
Test size : 10000


In [9]:
# Bag of Words
bow = CountVectorizer(max_features=5000)
X_train_bow = bow.fit_transform(X_train)
X_test_bow  = bow.transform(X_test)
print("BoW shape   :", X_train_bow.shape)

# TF-IDF
tfidf = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)
print("TF-IDF shape:", X_train_tfidf.shape)

BoW shape   : (40000, 5000)
TF-IDF shape: (40000, 5000)


## Step 6 – Model Building & Evaluation

In [10]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Naive Bayes'        : MultinomialNB(),
    'Decision Tree'      : DecisionTreeClassifier()
}

# Evaluate all models on TF-IDF
print("=" * 50)
print("Results with TF-IDF")
print("=" * 50)
for name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    pred = model.predict(X_test_tfidf)
    print(f"\n{name}")
    print(f"Accuracy: {accuracy_score(y_test, pred):.4f}")
    print(classification_report(y_test, pred, target_names=['Negative','Positive']))

Results with TF-IDF

Logistic Regression
Accuracy: 0.8890
              precision    recall  f1-score   support

    Negative       0.90      0.87      0.89      4961
    Positive       0.88      0.90      0.89      5039

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000


Naive Bayes
Accuracy: 0.8548
              precision    recall  f1-score   support

    Negative       0.86      0.85      0.85      4961
    Positive       0.85      0.86      0.86      5039

    accuracy                           0.85     10000
   macro avg       0.85      0.85      0.85     10000
weighted avg       0.85      0.85      0.85     10000


Decision Tree
Accuracy: 0.7138
              precision    recall  f1-score   support

    Negative       0.71      0.72      0.71      4961
    Positive       0.72      0.71      0.71      5039

    accuracy                           0.71     10000
   macro

In [11]:
# Evaluate all models on BoW
print("=" * 50)
print("Results with Bag of Words")
print("=" * 50)
for name, model in models.items():
    model.fit(X_train_bow, y_train)
    pred = model.predict(X_test_bow)
    print(f"{name} (BoW) Accuracy: {accuracy_score(y_test, pred):.4f}")

Results with Bag of Words
Logistic Regression (BoW) Accuracy: 0.8739
Naive Bayes (BoW) Accuracy: 0.8518
Decision Tree (BoW) Accuracy: 0.7087


## Step 7 – Comparison & Insights

**Best Preprocessing Steps:**
- Removing HTML tags was critical for IMDb data (reviews contain `<br />` tags)
- Stopword removal reduced noise significantly
- Lemmatization kept readable word forms better than stemming

**Best Vectorizer: TF-IDF**
- TF-IDF outperforms BoW because it weights rare but important words higher
- BoW treats all words equally which adds noise from common words

**Best Model: Logistic Regression**
- Logistic Regression performs best on sparse high-dimensional text features
- Naive Bayes is fastest but slightly less accurate
- Decision Tree overfits on text data and gives lowest accuracy

**Trade-offs:**
- Naive Bayes: fastest training, good for quick results
- Logistic Regression: best accuracy, slightly slower
- Decision Tree: most interpretable, but weakest performer on text